# Rwanda Hustle - ML Model Training

Complete training pipeline with confusion matrix analysis.

**Dataset:** 10,000 Rwandan businesses  
**Models:** Random Forest, Gradient Boosting, SVM  
**Goal:** Predict business success status

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
import pickle
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print('Libraries imported')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('rwanda_side_hustle_10000.csv')
print(f'Dataset: {len(df)} records, {len(df.columns)} columns')
df.head()

## 3. Explore Data

In [ ]:
print('Target Distribution:')
print(df['success_status'].value_counts())
print()
print('Percentage:')
print(df['success_status'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df['success_status'].value_counts().plot.pie(
    ax=ax1, autopct='%1.1f%%', startangle=90
)
ax1.set_title('Status Distribution')

df['success_status'].value_counts().plot.bar(ax=ax2)
ax2.set_title('Status Count')
ax2.set_xlabel('Status')

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Encode categorical features
le_gender = LabelEncoder()
le_education = LabelEncoder()
le_location = LabelEncoder()

df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['education_level_encoded'] = le_education.fit_transform(df['education_level'])
df['location_type_encoded'] = le_location.fit_transform(df['location_type'])

# Create features
df['has_training'] = (df['has_training'] == 'Yes').astype(int)
df['business_plan'] = (df['business_plan'] == 'Yes').astype(int)
df['access_to_internet'] = (df['access_to_internet'] == 'Yes').astype(int)
df['readiness_score'] = df['has_training'] + df['business_plan'] + df['access_to_internet']

df['profit_per_hour'] = df['monthly_profit (RWF)'] / (df['hours_per_week'] * 4.33)
df['profit_per_hour'] = df['profit_per_hour'].fillna(0)

df['roi_months'] = df['initial_capital (RWF)'] / (df['monthly_profit (RWF)'] + 1)
df['roi_months'] = df['roi_months'].fillna(999).clip(upper=999)

print('Features created')

In [ ]:
# Select features
feature_columns = [
    'age', 'gender_encoded', 'education_level_encoded', 'location_type_encoded',
    'hours_per_week', 'initial_capital (RWF)', 'monthly_profit (RWF)',
    'readiness_score', 'profit_per_hour', 'roi_months', 'access_to_internet'
]

X = df[feature_columns]
y = df['success_status']

# Encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f'Features: {len(feature_columns)}')
print(f'Classes: {list(le_target.classes_)}')

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')

## 6. Train Models

In [ ]:
# Random Forest
print('Training Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=10,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'Random Forest accuracy: {rf_acc*100:.2f}%')

In [ ]:
# Gradient Boosting
print('Training Gradient Boosting...')
gb_model = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.1, max_depth=5,
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_acc = accuracy_score(y_test, gb_pred)
print(f'Gradient Boosting accuracy: {gb_acc*100:.2f}%')

In [ ]:
# SVM
print('Training SVM...')
svm_model = SVC(C=10.0, kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'SVM accuracy: {svm_acc*100:.2f}%')

## 7. Confusion Matrices

In [ ]:
# Create confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices - Model Comparison', fontsize=16, fontweight='bold')

models = [
    ('Random Forest', rf_pred, rf_acc),
    ('Gradient Boosting', gb_pred, gb_acc),
    ('SVM', svm_pred, svm_acc)
]

for idx, (name, predictions, accuracy) in enumerate(models):
    cm = confusion_matrix(y_test, predictions)
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=True,
        xticklabels=le_target.classes_,
        yticklabels=le_target.classes_,
        ax=axes[idx], square=True
    )
    
    axes[idx].set_title(f'{name}\nAccuracy: {accuracy*100:.2f}%', fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print('Confusion matrices saved: confusion_matrices.png')

## 8. Model Comparison

In [ ]:
# Compare models
results = []
for name, predictions, _ in models:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, average='weighted'
    )
    accuracy = accuracy_score(y_test, predictions)
    results.append({
        'Model': name,
        'Accuracy': f'{accuracy*100:.2f}%',
        'Precision': f'{precision*100:.2f}%',
        'Recall': f'{recall*100:.2f}%',
        'F1-Score': f'{f1*100:.2f}%'
    })

results_df = pd.DataFrame(results)
print('Model Performance Comparison:')
results_df

In [ ]:
# Accuracy bar chart
accuracies = [rf_acc*100, gb_acc*100, svm_acc*100]
model_names = ['Random Forest', 'Gradient Boosting', 'SVM']

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, accuracies, color=['#2563eb', '#10b981', '#f59e0b'], alpha=0.8)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison', fontweight='bold', fontsize=14)
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Comparison chart saved: model_comparison.png')

## 9. Save Models

In [ ]:
# Save best model (Random Forest)
with open('models/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

with open('models/rf_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('models/rf_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f)

print('Models saved in models/ folder')
print('Training complete!')

## Summary

**Best Model:** Random Forest  
**Dataset:** 10,000 businesses  
**Test Size:** 2,000 samples  

**Results:**
- Random Forest: Highest accuracy
- Gradient Boosting: Good performance
- SVM: Lower accuracy

**Output Files:**
- confusion_matrices.png
- model_comparison.png
- models/random_forest_model.pkl